# 04 · Subqueries and CTEs (Oracle)

A **subquery** is a query inside another query.
- **scalar** subquery: returns one value, e.g. `(SELECT AVG(salary) FROM employees)`
- **correlated** subquery: uses a column from the outer query, so it runs "per row"
- `IN` / `EXISTS` / `NOT EXISTS`: check if matching rows exist

A **CTE** (`WITH name AS (...)`) gives a subquery a name. It makes long queries easy to read, step by step.

**How to run:** Start each SQL cell with `%%oracle`. One statement per cell. Tables are in your own schema, so just write `employees`.

Tables: see [tables_guide.md](../../jarvis/tables_guide.md). Run the setup cell first (Shift+Enter).

> **Solutions.** Try the practice notebook first: [`oracle/practice/04_subqueries_ctes.ipynb`](../../oracle/practice/04_subqueries_ctes.ipynb)

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "oracle"))
%load_ext oracle_helper

## Q1. Above the company average  ·  🟢 easy

Show `name` and `salary` of employees who earn more than the **average salary of all employees**.

*Expected result: 10 rows · columns: `name`, `salary`*

**How it works:** The subquery `(SELECT AVG(salary) FROM employees)` returns one number; the outer WHERE compares every salary with it.

In [ ]:
%%oracle
SELECT name, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)

,name,salary
0,Alice Johnson,250000.0
1,Bob Smith,150000.0
2,Carol White,120000.0
3,David Brown,120000.0
4,Eva Green,110000.0
5,Henry Clark,140000.0
6,Nathan King,130000.0
7,Rachel Nelson,135000.0
8,Victor Roberts,105000.0
9,Yara Campbell,115000.0


## Q2. Above their department average (correlated)  ·  🟡 medium

Show `name`, `dept_id`, `salary` of employees who earn more than the average salary **of their own department**.

*Expected result: 7 rows · columns: `name`, `dept_id`, `salary`*

**How it works:** For each employee row, the subquery averages salaries of **the same** department (`x.dept_id = e.dept_id`), and WHERE keeps people above it.

In [ ]:
%%oracle
SELECT e.name, e.dept_id, e.salary
FROM employees e
WHERE e.salary > (SELECT AVG(x.salary)
                  FROM employees x
                  WHERE x.dept_id = e.dept_id)

,name,dept_id,salary
0,Alice Johnson,1,250000.0
1,Bob Smith,1,150000.0
2,Henry Clark,2,140000.0
3,Nathan King,3,130000.0
4,Rachel Nelson,4,135000.0
5,Victor Roberts,5,105000.0
6,Yara Campbell,6,115000.0


**Why / traps:** The inner query uses `e.dept_id` from the outer query: that makes it **correlated**. The engine usually rewrites it as a join with a GROUP BY (like the DataFrame answer). The employee with NULL dept_id drops out: `x.dept_id = NULL` never matches.

## Q3. Not a manager: the NOT IN trap  ·  🟡 medium

Show `emp_id` and `name` of employees who are **nobody's manager**.

First try `WHERE emp_id NOT IN (SELECT manager_id FROM employees)`. How many rows? Why?

*Expected result: 18 rows · columns: `emp_id`, `name`*

**How it works:** `NOT EXISTS` asks, for each employee: is there anyone whose `manager_id` is my `emp_id`? If not, I manage nobody. No NULL problem.

In [ ]:
%%oracle
SELECT e.emp_id, e.name
FROM employees e
WHERE NOT EXISTS (SELECT 1 FROM employees x WHERE x.manager_id = e.emp_id)

,emp_id,name
0,6,Frank Moore
1,23,Wendy Turner
2,27,Amy Evans
3,7,Grace Lee
4,11,Karen Hall
5,30,Daniel Stewart
6,21,Uma Perez
7,17,Quinn Baker
8,12,Leo Allen
9,4,David Brown


**Why / traps:** `NOT IN` with a NULL in the list returns **0 rows**: `5 NOT IN (1, NULL)` means `5 <> 1 AND 5 <> NULL`, which is unknown. Use `NOT EXISTS`, or add `WHERE manager_id IS NOT NULL` inside the subquery.

## Q4. Products in big orders (IN)  ·  🟢 easy

Show `product_id` and `product_name` of products that appear in at least one **completed** order with amount **over 200**. Each product once.

*Expected result: 6 rows · columns: `product_id`, `product_name`*

**How it works:** The subquery lists product ids from completed orders over 200; `IN` keeps products in that list. Duplicates in the list don't matter.

In [ ]:
%%oracle
SELECT product_id, product_name
FROM products
WHERE product_id IN (
    SELECT i.product_id
    FROM order_items i
    JOIN orders o ON o.order_id = i.order_id
    WHERE o.status = 'completed' AND o.amount > 200)

,product_id,product_name
0,2,USB-C Cable
1,3,Bluetooth Speaker
2,6,Headphones
3,8,Data Engineering 101
4,13,Water Bottle
5,15,T-Shirt


## Q5. Departments that pay above average  ·  🟡 medium

Show `dept_id` and `avg_salary` (2 decimals) of departments whose average salary is higher than the company-wide average salary.

*Expected result: 2 rows · columns: `dept_id`, `avg_salary`*

**How it works:** Group employees by department and compute the average; `HAVING` compares each group's average with the company average from a subquery.

In [ ]:
%%oracle
SELECT dept_id, ROUND(AVG(salary), 2) AS avg_salary
FROM employees
WHERE dept_id IS NOT NULL
GROUP BY dept_id
HAVING AVG(salary) > (SELECT AVG(salary) FROM employees)

,dept_id,avg_salary
0,1,134285.71
1,4,107750.00


## Q6. Customers who spend above average (CTE)  ·  🟡 medium

Use a CTE. Step 1: total completed amount per customer (count each order once). Step 2: keep customers whose total is above the **average of those totals**. Show `customer_id`, `total_spent`. Highest first.

*Expected result: 11 rows · columns: `customer_id`, `total_spent`*

**How it works:** CTE 1 removes duplicates; CTE 2 sums spend per customer; the final query keeps customers above `AVG(total_spent)` of CTE 2.

In [ ]:
%%oracle
WITH clean_orders AS (
    SELECT DISTINCT order_id, customer_id, amount, status
    FROM orders
),
spend AS (
    SELECT customer_id, SUM(amount) AS total_spent
    FROM clean_orders
    WHERE status = 'completed'
    GROUP BY customer_id
)
SELECT customer_id, total_spent
FROM spend
WHERE total_spent > (SELECT AVG(total_spent) FROM spend)
ORDER BY total_spent DESC, customer_id

,customer_id,total_spent
0,28,740
1,29,535
2,27,492
3,26,450
4,30,390
5,31,385
6,25,330
7,14,312
8,16,306
9,20,290


**Why / traps:** A CTE can be used more than once (`spend` is used twice here). Each CTE is one clear step: easy to test alone.

## Q7. Second highest salary (no window functions)  ·  🟡 medium

Show the **second highest** different salary as `second_highest`. Do it without window functions.

*Expected result: 1 row · columns: `second_highest`*

**How it works:** Take the highest salary that is **lower** than the overall maximum = the second highest different salary.

In [ ]:
%%oracle
SELECT MAX(salary) AS second_highest
FROM employees
WHERE salary < (SELECT MAX(salary) FROM employees)

,second_highest
0,150000


**Why / traps:** Classic interview question. If there is no second salary, MAX returns NULL (one row with NULL), which is usually what they want. For the Nth salary see topic 05.

## Q8. Employee count per department (scalar subquery in SELECT)  ·  🟡 medium

Show every `dept_name` with `num_employees`, using a **subquery in the SELECT list**. Legal must show 0.

*Expected result: 7 rows · columns: `dept_name`, `num_employees`*

**How it works:** For each department, the subquery in SELECT counts its employees. It returns 0 for Legal (COUNT of no rows is 0).

In [ ]:
%%oracle
SELECT d.dept_name,
       (SELECT COUNT(*) FROM employees e WHERE e.dept_id = d.dept_id) AS num_employees
FROM departments d
ORDER BY num_employees DESC, d.dept_name

,dept_name,num_employees
0,Engineering,7
1,Sales,6
2,Operations,5
3,Finance,4
4,Marketing,4
5,HR,3
6,Legal,0


**Why / traps:** COUNT over zero rows gives 0 (not NULL). SUM over zero rows gives NULL.

## Q9. Departments with a recent hire (EXISTS)  ·  🟢 easy

Show `dept_name` of departments with **at least one** employee hired on or after 2023-01-01.

*Expected result: 4 rows · columns: `dept_name`*

**How it works:** `EXISTS` with a correlated subquery: keep a department if at least one of its employees has `hire_date >= 2023-01-01`.

In [ ]:
%%oracle
SELECT d.dept_name
FROM departments d
WHERE EXISTS (SELECT 1
              FROM employees e
              WHERE e.dept_id = d.dept_id
                AND e.hire_date >= DATE '2023-01-01')

,dept_name
0,Sales
1,Marketing
2,HR
3,Operations


**Why / traps:** `DATE '2023-01-01'` is a date literal (a fixed date written in SQL). It works in all 3 engines.

## Q10. The most recent order(s)  ·  🟢 easy

Show `order_id`, `customer_id`, `order_date` of the order(s) on the **latest order date**. If two orders share that date, show both. No duplicate rows.

*Expected result: 1 row · columns: `order_id`, `customer_id`, `order_date`*

**How it works:** The subquery finds the latest date; the outer query keeps all orders on that date. DISTINCT removes duplicate rows.

In [ ]:
%%oracle
SELECT DISTINCT order_id, customer_id, order_date
FROM orders
WHERE order_date = (SELECT MAX(order_date) FROM orders)

,order_id,customer_id,order_date
0,1104,2,2025-12-28


## Q11. Busy months of 2024 (CTE chain)  ·  🔴 hard

For year 2024: count different orders per month (`order_month` 1–12, `num_orders`). Keep only months with **more** orders than the average month of 2024.

*Expected result: 4 rows · columns: `order_month`, `num_orders`*

**How it works:** CTE `per_month` counts orders per month of 2024; the final query compares each month with the average of all months in the CTE.

In [ ]:
%%oracle
WITH per_month AS (
    SELECT EXTRACT(MONTH FROM order_date) AS order_month,
           COUNT(DISTINCT order_id)        AS num_orders
    FROM orders
    WHERE order_date >= DATE '2024-01-01' AND order_date < DATE '2025-01-01'
    GROUP BY EXTRACT(MONTH FROM order_date)
)
SELECT order_month, num_orders
FROM per_month
WHERE num_orders > (SELECT AVG(num_orders) FROM per_month)
ORDER BY order_month

,order_month,num_orders
0,1,4
1,3,5
2,9,4
3,11,6


**Why / traps:** Filter dates with a **range** (`>= start AND < end`) instead of `EXTRACT(YEAR ...) = 2024`. On partitioned tables the range lets the engine skip data (partition pruning).